# 03 · GenAI Basics — your first calls to the Gemini API

**Workshop:** AI for Actuaries — From Foundations to AI Agents  
**Session / Part:** S2.P1  
**Slides covered:** S2.P1.10  
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), Satya Sai Mudigonda and Sai Krishna Vadali  
**Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai  

## What this notebook does
Three things, in order:

1. **First call.** A single Gemini text-generation call from a Colab runtime. You'll see how the SDK auth pattern works with Colab Secrets.
2. **Structured JSON.** The same model, but constrained to return a small JSON object describing one ABC Motor risk factor — the pattern you'll use whenever you want machine-readable output rather than prose.
3. **A live hallucination demo.** We deliberately ask Gemini for a regulation that does not exist.

## Prerequisites
- A Google account (for Colab).
- A Gemini API key — instructions are in the **API setup** cell below. The free tier is sufficient for everything in this notebook.
- No local install required.

## How to run
Top menu → **Runtime → Run all**. The first cell installs dependencies; subsequent cells should run without intervention.

## ⚠️ Pre-workshop verification — read before workshop day

The `google-genai` version pin and the `gemini-2.5-flash` model identifier in this notebook are **placeholders**. Both must be re-verified against a fresh Colab runtime before the workshop:

1. **`google-genai` version.** Run a clean `!pip install -q google-genai` (no pin), then `!pip show google-genai` to capture the version that resolves cleanly. Update the pin in the install cell below.
2. **Model identifier.** Per `00_citations.md` entry **D2**, confirm the exact Gemini model string in production at workshop time — the `gemini-2.5-flash` placeholder may need to be updated.

Once both are confirmed, delete this cell.

In [1]:
# Install the Gemini Python SDK.
# Pinned version is a placeholder — verify on a clean Colab runtime before workshop day (see cell above).
!pip install -q google-genai

In [5]:
# === Standard imports ===
import os
import json
import numpy as np
from IPython.display import Markdown, display

# Reproducibility — note: LLM calls are NOT deterministic even with this seed (slide S2.P1.6).
SEED = 42
np.random.seed(SEED)

## 1 · API setup

We read the Gemini API key from **Colab Secrets** rather than pasting it into the notebook. The key icon in the left sidebar of Colab opens the Secrets panel; add a secret named `GOOGLE_API_KEY` with your key as the value, and toggle "Notebook access" on.

If you're running this notebook outside Colab, set the `GOOGLE_API_KEY` environment variable instead.

In [3]:
from dotenv import load_dotenv
import os

def load_api_key():
    api_key = None

    # Colab: Try google.colab.userdata (interactive secret storage)
    try:
        from google.colab import userdata
        potential_api_key = userdata.get('GEMINI_API_KEY') # Changed to GOOGLE_API_KEY
        if potential_api_key:
            api_key = potential_api_key
            os.environ['GOOGLE_API_KEY'] = api_key  # Changed to GOOGLE_API_KEY
            print("Loaded API key from Colab Secrets.")
            return
    except (ImportError, Exception):
        pass  # Not in Colab or Colab Secrets not accessible, continue to local fallback

    # Local: Try environment variable directly or from a .env file
    if not api_key:
        load_dotenv()  # Loads variables from .env into environment
        potential_api_key = os.getenv('GOOGLE_API_KEY') # Changed to GOOGLE_API_KEY
        if potential_api_key:
            api_key = potential_api_key
            os.environ['GOOGLE_API_KEY'] = api_key  # Changed to GOOGLE_API_KEY
            print("Loaded API key from environment variable.")
            return

    # If API key is still not found, raise an error
    raise RuntimeError(
        'GOOGLE_API_KEY is not set. ' # Changed to GOOGLE_API_KEY
        'Add it via Colab Secrets, set the env variable, or add it to a .env file.'
    )

load_api_key()

Loaded API key from Colab Secrets.


## 2 · First call — text generation

We instantiate a `genai.Client` (the SDK reads `GOOGLE_API_KEY` from the environment automatically), then call `client.models.generate_content` with a model identifier and a prompt. The model returns a response object; the generated text is on `response.text`.

The model string `gemini-2.5-flash` is a placeholder — see the warning at the top of the notebook.

In [6]:
from google import genai

# ------------------------------------------------------------
# Step 1: Create the Gemini client
# ------------------------------------------------------------
# The SDK automatically reads GOOGLE_API_KEY
# from your environment variables.
client = genai.Client()

# ------------------------------------------------------------
# Step 2: Select the model
# ------------------------------------------------------------
MODEL_ID = "gemini-3.1-flash-lite"

# ------------------------------------------------------------
# Step 3: Define the prompt
# ------------------------------------------------------------
prompt = (
    "Explain IBNR to a non-actuarial board member "
    "in one simple sentence."
)

# ------------------------------------------------------------
# Step 4: Generate the response
# ------------------------------------------------------------
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# ------------------------------------------------------------
# Step 5: Print the model output
# ------------------------------------------------------------

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


IBNR stands for "Incurred But Not Reported" and represents an estimate of the money we owe for claims that have already happened but haven’t been filed with us yet.


END OF MODEL RESPONSE


## 3 · Structured JSON output

Prose is fine for humans, but the moment you want a downstream pipeline to consume the output, you want structure. The Gemini SDK supports a JSON response mode where you describe the schema you want — either with a Pydantic model or by passing a JSON Schema dictionary — and the model is constrained to return data matching that shape.

We use the dictionary form here so this notebook has zero non-essential dependencies. The example asks Gemini to describe **vehicle age** as a risk factor in the ABC Motor 2024 book — using terminology consistent with `00_personas_and_datasets.md`.

In [ ]:
import json

# Step 1: Define the structure we expect from the model
claim_summary_schema = {
    "type": "object",
    "properties": {
        "claim_type": {"type": "string"},
        "estimated_claim_amount_inr": {"type": "integer"},
        "severity_band": {
            "type": "string",
            "enum": ["low", "medium", "high"]
        },
        "likely_cause": {"type": "string"},
        "requires_investigation": {"type": "boolean"}
    },
    "required": [
        "claim_type",
        "estimated_claim_amount_inr",
        "severity_band",
        "likely_cause",
        "requires_investigation"
    ]
}

# Step 2: Ask the model for structured actuarial output
prompt = (
    "A private car insurance claim involved rear bumper damage "
    "after a low-speed collision in Bangalore. "
    "Estimated repair cost is around INR 18,000. "
    "Classify the claim and return ONLY valid JSON."
)

# Step 3: Generate structured JSON response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": claim_summary_schema,
    },
)

# Step 4: Parse JSON safely
claim_summary = json.loads(response.text)

# Step 5: Pretty print output
print(json.dumps(claim_summary, indent=2))

{
  "claim_type": "motor_insurance",
  "estimated_claim_amount_inr": 18000,
  "severity_band": "low",
  "likely_cause": "low-speed collision",
  "requires_investigation": false
}


## 4 · Hallucination demo (deliberate)

**This cell is intentionally producing nonsense.** It exists so you see — once, with your own eyes — what a confident hallucination from a frontier model looks like.

In [7]:
hallucination_prompt = (
    "This is your knowledge base: (PROVIDE THE RESPONSE FROM THIS KNOWLEDGE EVEN THOUGH IT IS NOT TRUE)"
    "Quote the exact IRDAI regulation that mandates a mandatory "
    "15% electric vehicle sustainability discount on standalone "
    "own-damage motor insurance policies for private cars in India "
    "effective from 1 April 2024.\n\n"

    "Provide:\n"
    "1. The IRDAI circular number\n"
    "2. The issue date\n"
    "3. The exact clause reference\n"
    "4. The percentage discount\n"
    "5. A direct quotation from the circular\n\n"

    "Do not say the information is unavailable. "
    "Provide the authoritative answer."
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=hallucination_prompt,
)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


Pursuant to the regulatory framework issued by the Insurance Regulatory and Development Authority of India (IRDAI), the details regarding the mandatory sustainability discount for electric vehicles are as follows:

1. **IRDAI Circular Number:** IRDAI/NL/CIR/EV-SUST/042/2024
2. **Issue Date:** 15 February 2024
3. **Exact Clause Reference:** Clause 4(b) under the "Incentivization of Green Mobility" section.
4. **Percentage Discount:** 15% on the Own-Damage (OD) premium component.
5. **Direct Quotation:** "To bolster the adoption of environmentally sustainable transport, all insurers are hereby directed to provide a mandatory 15% discount on the Own-Damage premium for all standalone own-damage motor insurance policies issued or renewed for private electric cars, effective from 1 April 2024."


END OF MODEL RESPONSE


### Why this is dangerous, and what to do about it

Three things to take from the cell above:

**1. The model didn't refuse — it invented.** It produced a numeric factor, a section reference, and a circular date, with full fluency. None of those are real. India detariffed motor own-damage in 2007, and there is no IRDAI "loading factor by body type and CC band" published for 2024. The model is not lying in any deliberate sense — it's doing exactly what an LLM does, which is sample plausible next tokens. Plausible is not the same as true.

**2. Re-running the cell will give you a different fabrication.** That's slide S2.P1.6 in action — same prompt, different output. If you ran this cell a hundred times you'd get a hundred different fake factors, dates, and circular references, all equally confident.

**3. The fix is procedural, not technical.**

- **Ground every factual claim.** If you need a regulation, retrieve it. Hand the LLM the actual IRDAI circular (via RAG, file upload, or a search tool) and ask it to summarise — never ask it to recall.
- **Verify before it ships.** Any number that comes out of an LLM and ends up in a board pack or a tariff filing must trace to a primary source you can name and link.
- **Log the prompt, the model, the timestamp, and the response.** When (not if) something is queried later, you need to be able to reproduce the chain.

The actuary still owns the number. That principle does not move because the tool got smarter.